# 53 — News Relevance Filtering
Filter raw news before AI summarisation.

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def normalise_date_col(df):
    for c in df.columns:
        if c.lower() in {"date","day","datetime","timestamp","time","time_utc","publishedat","published_at"}:
            out = df.copy()
            out["date"] = pd.to_datetime(out[c], errors="coerce").dt.normalize()
            return out, c
    return df.copy(), None

def list_output_files():
    rows = []
    if not OUTPUTS.exists():
        return pd.DataFrame(columns=["path","size_bytes","sha256"])
    for p in sorted(OUTPUTS.rglob("*")):
        if p.is_file():
            rows.append({
                "path": str(p.relative_to(PROJECT_ROOT)),
                "size_bytes": p.stat().st_size,
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

run_cfg = load_yaml(CONFIGS / "run.yml")
phase_cfg = load_yaml(CONFIGS / "phase50.yml")
sites_df = pd.DataFrame(run_cfg.get("sites", []))
phase_sites = pd.DataFrame(phase_cfg.get("additional_sites", []))
if not phase_sites.empty:
    existing = set(sites_df.get("id", pd.Series(dtype=str)).astype(str).tolist()) if not sites_df.empty else set()
    phase_sites = phase_sites[~phase_sites["id"].astype(str).isin(existing)]
    sites_df = pd.concat([sites_df, phase_sites], ignore_index=True)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])
date_from = run_cfg.get("run", {}).get("date_from", "2025-01-01")
date_to = run_cfg.get("run", {}).get("date_to", "2025-01-31")

In [ ]:
step = "53_news_relevance_filtering"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

news_cfg = load_yaml(CONFIGS / "news_filters.yml")
include_terms = [str(x).lower() for x in news_cfg.get("include_terms", [])]
geo_terms = [str(x).lower() for x in news_cfg.get("geo_terms", [])]
exclude_terms = [str(x).lower() for x in news_cfg.get("exclude_terms", [])]
min_score = int(news_cfg.get("minimum_relevance_score", 2))

src = OUTPUTS / "03_news_context" / "news_articles.json"
if src.exists():
    raw = load_json(src)
    rows = []
    for item in raw:
        title = str(item.get("title", ""))
        desc = str(item.get("description", ""))
        pub = item.get("publishedAt") or item.get("published_at") or item.get("date")
        text = f"{title} {desc}".lower()
        include_hits = sum(term in text for term in include_terms)
        geo_hits = sum(term in text for term in geo_terms)
        exclude_hits = sum(term in text for term in exclude_terms)
        score = include_hits * 2 + geo_hits - exclude_hits * 2
        rows.append({
            "published_at": pub,
            "title": title,
            "description": desc,
            "relevance_score": score,
            "keep": score >= min_score,
        })
    df = pd.DataFrame(rows)
else:
    df = pd.DataFrame(columns=["published_at","title","description","relevance_score","keep"])

kept = df[df["keep"]].copy() if not df.empty else df.copy()
rejected = df[~df["keep"]].copy() if not df.empty else df.copy()

df.to_csv(out / "news_relevance_scored.csv", index=False)
kept.to_csv(out / "news_relevance_kept.csv", index=False)
rejected.to_csv(out / "news_relevance_rejected.csv", index=False)

summary = {
    "raw_article_count": int(len(df)),
    "kept_article_count": int(len(kept)),
    "rejected_article_count": int(len(rejected)),
}
write_json(out / "news_relevance_summary.json", summary)

manifest = manifest_base(step, [CONFIGS / "news_filters.yml"])
for p in [out / "news_relevance_scored.csv", out / "news_relevance_kept.csv", out / "news_relevance_summary.json"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
write_json(out / "manifest.json", manifest)
display(kept.head(20))
print(summary)